In [1]:
from numba import njit, prange
import numpy as np
import faiss

data = np.load("/home/dhem/workspace/2024.3/data/save/train_test-0-0-0-0.npz")

x = data["x"]
y = data["y"]
w = data["w"]
coor = data["coor"]
name = data["name"]

In [2]:
(num_sample, dim_sample) = x.shape
res = faiss.StandardGpuResources()
flat_config = faiss.GpuIndexFlatConfig()
flat_config.device = 0
index = faiss.GpuIndexFlatL2(res, dim_sample, flat_config)
index.add(x)

In [3]:
b = x.copy()
min_number = 40
distances, indices = index.search(b, min_number)

var_y = np.var(np.einsum("ij,i->ij", y[indices], x[:, 0] * w), axis=1)
argsort_ = np.argsort(var_y)[::-1][:100]
print(np.einsum("ij,i->ij", y[indices], x[:, 0] * w)[argsort_])

[[-4.73994617e-05 -4.79877250e-05 -4.73927406e-05 -4.45448024e-05
  -4.45735080e-05 -4.67054803e-05 -4.56267719e-05 -4.16096525e-05
  -3.35190768e-05 -3.31722212e-05 -2.98717720e-05 -2.49402721e-05
  -2.42643075e-05 -3.98025163e-05 -2.82803005e-05 -2.44519279e-05
  -4.41069759e-05 -4.84284252e-05 -3.39282014e-05 -2.42730456e-05
  -3.91499579e-05 -3.99661444e-05 -5.10057070e-05 -3.30811642e-05
  -5.42751939e-06 -4.36324458e-05 -5.20924054e-06 -4.58227533e-05
  -3.10102296e-05 -2.41602779e-05 -2.45849827e-05 -7.58285836e-06
  -7.37800324e-06 -4.31193865e-06 -4.12936163e-06 -4.79674947e-05
  -4.95478565e-05 -3.92065099e-05 -2.41969374e-05 -3.79003639e-05]
 [-3.98547454e-05 -3.97171113e-05 -3.18371344e-05 -3.15820074e-05
  -3.96698706e-05 -3.80944748e-05 -4.02858320e-05 -4.23926696e-05
  -4.07865891e-05 -4.11287937e-05 -2.32616754e-05 -2.31590957e-05
  -4.24628725e-05 -4.13210107e-05 -4.21187019e-05 -2.21457396e-05
  -4.24835339e-05 -3.83793813e-05 -3.75894916e-05 -4.08790307e-05
  -2.2331

In [4]:
distance = np.sum(
    np.transpose(
        np.transpose(x[indices[argsort_]], axes=(0, 2, 1)) - x[argsort_][:, :, None],
        axes=(0, 2, 1),
    ) ** 2,
    axis=2,
)
# print(x[argsort_][:, :, None].shape)
# print(np.transpose(x[indices[argsort_]], axes=(0, 2, 1)).shape)
energy = np.einsum("ij,i->ij", y[indices[argsort_]], (x[:, 0] * w)[argsort_])
print(y[indices[argsort_]] - y[argsort_][:, None])
print(energy)

[[ 0.00000000e+00 -2.58797630e-04  2.95681507e-06  1.25586444e-03
   1.24323587e-03  3.05306681e-04  7.79868227e-04  2.54713955e-03
   6.10646677e-03  6.25906070e-03  7.71104374e-03  9.88058299e-03
   1.01779635e-02  3.34216196e-03  8.41118771e-03  1.00954227e-02
   1.44847963e-03 -4.52677059e-04  5.92647853e-03  1.01741193e-02
   3.62924525e-03  3.27017624e-03 -1.58651347e-03  6.29911985e-03
   1.84649254e-02  1.65724204e-03  1.85609539e-02  6.93649136e-04
   7.21019640e-03  1.02237297e-02  1.00368872e-02  1.75167164e-02
   1.76068393e-02  1.89557084e-02  1.90360304e-02 -2.49897580e-04
  -9.45154020e-04  3.60436602e-03  1.02076020e-02  4.17898534e-03]
 [ 0.00000000e+00  6.74272967e-05  3.92784792e-03  4.05283530e-03
   9.05706067e-05  8.62361024e-04 -2.11190401e-04 -1.24333550e-03
  -4.56512585e-04 -6.24159508e-04  8.12898701e-03  8.17924104e-03
  -1.27772807e-03 -7.18327065e-04 -1.10911801e-03  8.67568676e-03
  -1.28785015e-03  7.22784607e-04  1.10975357e-03 -5.01799989e-04
   8.5847

In [10]:
from matplotlib import pyplot as plt

color_dict = {
    "methane_cc-pVDZ_0-1_1_-0.2000": "#004D40",
    "methane_cc-pVDZ_0-1_1_-0.1000": "#1A237E",
    "methane_cc-pVDZ_0-1_1_0.0000": "#7B1FA2",
    "methane_cc-pVDZ_0-1_1_0.1000": "#B71C1C",
    "methane_cc-pVDZ_0-1_1_0.2000": "#FF6F00",
}

plt.rcParams["figure.figsize"] = np.array([3, 3]) * 520 / 72

f, axes = plt.subplots(10, 10)
axes = axes.reshape(10, 10)

begin_y = 0.025
end_y = 0.95
int_y = 0.0
begin_x = 0.025
end_x = 0.95
int_x = 0.0
end_x += int_x
end_y += int_y

shapexy = np.shape(axes)
inter_x = np.linspace(begin_x, end_x, shapexy[1] + 1)
inter_y = np.linspace(begin_y, end_y, shapexy[0] + 1)

delta_x = inter_x[1] - inter_x[0] - int_x
delta_y = inter_y[1] - inter_y[0] - int_y

for i in range(shapexy[0]):
    for j in range(shapexy[1]):
        axes[i][j].set_position(
            [
                inter_x[j],
                inter_y[i],
                inter_x[j + 1] - inter_x[j] - int_x,
                inter_y[i + 1] - inter_y[i] - int_y,
            ]
        )
        axes[i][j].xaxis.set_tick_params(
            direction="in", which="both", bottom=True, top=True
        )
        axes[i][j].yaxis.set_tick_params(
            direction="in", which="both", left=True, right=True
        )
        if i != 0:
            axes[i][j].set_xticks([])
        if j != 0:
            axes[i][j].set_yticks([])


for i in range(argsort_.shape[0]):
    axes_i, axes_j = np.unravel_index(i, (10, 10))
    for j in range(indices.shape[1]):
        axes[axes_i, axes_j].scatter(
            distance[i][j],
            np.abs(energy[i][j] - energy[i][0]) * 627.509,
            c=color_dict[name[indices[argsort_[i]]][j]],
        )
        axes[axes_i, axes_j].set_xlim(-0.005, 0.055)
        axes[axes_i, axes_j].set_ylim(-0.003, 0.033)
    axes[axes_i, axes_j].text(
        0.01,
        1-0.01,
        f"{i}",
        transform=axes[axes_i, axes_j].transAxes,
        va="top",
    )
plt.savefig("test.pdf", dpi=300)
plt.clf()

<Figure size 2166.67x2166.67 with 0 Axes>

In [20]:
np.exp(-0.001*10)

0.9900498337491681